In [1]:
import pandas as pd
import re
import nltk
from sklearn.model_selection import train_test_split
import math

In [2]:
nltk.download('stopwords')
nltk.download('punkt_tab')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [3]:
df = pd.read_csv('bbc-text (1).csv')
df

,category,text
0,tech,tv future in the hands of viewers with home th...
1,business,worldcom boss left books alone former worldc...
2,sport,tigers wary of farrell gamble leicester say ...
3,sport,yeading face newcastle in fa cup premiership s...
4,entertainment,ocean s twelve raids box office ocean s twelve...
...,...,...
2220,business,cars pull down us retail figures us retail sal...
2221,politics,kilroy unveils immigration policy ex-chatshow ...
2222,entertainment,rem announce new glasgow concert us band rem h...
2223,politics,how political squabbles snowball it s become c...


In [4]:
def data_preprocessing(text):
  text = text.lower()
  text = re.sub(r'[^\w\s]', '', text)
  token = nltk.word_tokenize(text)
  stopwords = nltk.corpus.stopwords.words('english')
  text = [t for t in token if t not in stopwords]
  return text

corpus = [data_preprocessing(txt) for txt in df['text']]

In [5]:
X = corpus
y = df['category']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, shuffle=True, random_state=42
)

In [6]:
def build_vocabulary(x):
  unique_words = []
  seen = set()
  for doc in x:
    for word in doc:
      if word not in seen:
        seen.add(word)
        unique_words.append(word)
  dct = {item: idx for idx, item in enumerate(unique_words)}
  return dct

dct = build_vocabulary(X_train)
dct

{'uganda': 0,
 'bans': 1,
 'vagina': 2,
 'monologues': 3,
 'authorities': 4,
 'banned': 5,
 'play': 6,
 'due': 7,
 'open': 8,
 'capital': 9,
 'kampala': 10,
 'weekend': 11,
 'ugandan': 12,
 'media': 13,
 'council': 14,
 'said': 15,
 'performance': 16,
 'would': 17,
 'put': 18,
 'promoted': 19,
 'glorified': 20,
 'acts': 21,
 'lesbianism': 22,
 'homosexuality': 23,
 'production': 24,
 'could': 25,
 'go': 26,
 'ahead': 27,
 'organisers': 28,
 'expunge': 29,
 'offending': 30,
 'parts': 31,
 'say': 32,
 'raises': 33,
 'awareness': 34,
 'sexual': 35,
 'abuse': 36,
 'women': 37,
 'promotes': 38,
 'illegal': 39,
 'unnatural': 40,
 'prostitution': 41,
 'hereby': 42,
 'ruling': 43,
 'show': 44,
 'controversial': 45,
 'sellout': 46,
 'around': 47,
 'world': 48,
 'explores': 49,
 'female': 50,
 'sexuality': 51,
 'strength': 52,
 'individual': 53,
 'telling': 54,
 'stories': 55,
 'parliamentarians': 56,
 'church': 57,
 'leaders': 58,
 'also': 59,
 'siding': 60,
 'new': 61,
 'vision': 62,
 'newspap

In [7]:
def BoW_vectors(text, dct):
  vector = [0] * len(dct)
  for i in text:
    if i in dct:
      vector[dct[i]] += 1
  return vector

X_train_vectors = [BoW_vectors(doc, dct) for doc in X_train]
X_test_vectors = [BoW_vectors(doc, dct) for doc in X_test]

In [8]:
def calculate_idf(X_train_vectors, dct):
    idf_dct = {}
    N = len(X_train_vectors)

    for i in dct:
        N_t = 0
        for j in X_train_vectors:
            if j[dct[i]] > 0:
                N_t += 1

        idf = math.log(N / N_t)
        idf_dct[i] = idf

    return idf_dct
idf_dct = calculate_idf(X_train_vectors, dct)

In [9]:
def calculate_tf(vector):
  total = sum(vector)
  if total == 0:
    return [0] * len(vector)
  return [count / total for count in vector]

def get_tfidf_vectors(vectors, dct, idf_dct):
  tfidf_vectors = []

  for vector in vectors:
    tf_vector = calculate_tf(vector)
    tfidf_vector = []
    for word, index in dct.items():
      tf_value = tf_vector[index]
      idf_value = idf_dct[word]
      tfidf_vector.append(tf_value * idf_value)

    tfidf_vectors.append(tfidf_vector)

  return tfidf_vectors
X_train_tfidf = get_tfidf_vectors(X_train_vectors, dct, idf_dct)
X_test_tfidf = get_tfidf_vectors(X_test_vectors, dct, idf_dct)

In [10]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

clf_bow = LogisticRegression(max_iter=1000)
clf_bow.fit(X_train_vectors, y_train)
pred_bow = clf_bow.predict(X_test_vectors)
print('BoW accuracy:', accuracy_score(y_test, pred_bow))

clf_tfidf = LogisticRegression(max_iter=1000)
clf_tfidf.fit(X_train_tfidf, y_train)
pred_tfidf = clf_tfidf.predict(X_test_tfidf)
print('TF-IDF accuracy:', accuracy_score(y_test, pred_tfidf))

print(classification_report(y_test, pred_tfidf))

BoW accuracy: 0.9655688622754491
TF-IDF accuracy: 0.9505988023952096
               precision    recall  f1-score   support

     business       0.91      0.96      0.93       164
entertainment       1.00      0.93      0.96       113
     politics       0.96      0.96      0.96       113
        sport       0.92      1.00      0.96       146
         tech       1.00      0.89      0.94       132

     accuracy                           0.95       668
    macro avg       0.96      0.95      0.95       668
 weighted avg       0.95      0.95      0.95       668



In [13]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC

clf_nb = MultinomialNB()
clf_nb.fit(X_train_tfidf, y_train)
print('Naive Bayes  :', accuracy_score(y_test, clf_nb.predict(X_test_tfidf)))

clf_svm = LinearSVC()
clf_svm.fit(X_train_tfidf, y_train)
print('Linear SVM   :', accuracy_score(y_test, clf_svm.predict(X_test_tfidf)))

Naive Bayes  : 0.905688622754491
Linear SVM   : 0.9775449101796407


In [14]:
import numpy as np

words = list(dct.keys())
tfidf_arr = np.array(X_train_tfidf)
y_arr = np.array(y_train)

def top10_words(category):
  rows = tfidf_arr[y_arr == category]
  mean_scores = rows.mean(axis=0)
  top_idx = mean_scores.argsort()[::-1][:10]
  return [(words[i], round(float(mean_scores[i]), 5)) for i in top_idx]

for category in sorted(set(y_arr)):
  print('\n', category)
  for word, score in top10_words(category):
    print(f'  {word:<15} {score}')


 business
  growth          0.009
  bank            0.00873
  economy         0.00853
  oil             0.00848
  market          0.00846
  shares          0.00822
  company         0.00788
  firm            0.00761
  us              0.00748
  prices          0.00732

 entertainment
  film            0.02288
  best            0.01144
  festival        0.01047
  band            0.01013
  music           0.00944
  show            0.00912
  awards          0.0091
  album           0.00906
  award           0.00895
  star            0.00796

 politics
  mr              0.01676
  labour          0.01576
  blair           0.0133
  election        0.01328
  party           0.01295
  government      0.01029
  howard          0.00925
  brown           0.00892
  lord            0.00772
  tory            0.00759

 sport
  england         0.01003
  game            0.00956
  cup             0.00919
  club            0.00834
  win             0.00803
  injury          0.00769
  rugby           0.00

In [15]:
total_tf = [0] * len(dct)
for vector in X_train_vectors:
  for i, count in enumerate(vector):
    total_tf[i] += count

In [16]:
mean_tf = sum(total_tf) / len(total_tf)
mean_idf = sum(idf_dct.values()) / len(idf_dct.values())
high_tf_low_idf = []
low_tf_high_idf = []
for word, ind in dct.items():
  if total_tf[ind] > mean_tf and idf_dct[word] < mean_idf:
    high_tf_low_idf.append(word)
  elif total_tf[ind] < mean_tf and idf_dct[word] > mean_idf:
    low_tf_high_idf.append(word)

In [17]:
print('High TF, low IDF:', len(high_tf_low_idf), 'words')
print(sorted(high_tf_low_idf, key=lambda w: total_tf[dct[w]], reverse=True)[:20])
print()
print('Low TF, high IDF:', len(low_tf_high_idf), 'words')
print(sorted(low_tf_high_idf, key=lambda w: idf_dct[w], reverse=True)[:20])

High TF, low IDF: 4385 words
['said', 'mr', 'would', 'also', 'new', 'people', 'us', 'year', 'one', 'could', 'last', 'first', 'two', 'world', 'time', 'government', 'uk', 'years', 'told', 'best']

Low TF, high IDF: 16388 words
['vagina', 'monologues', 'kampala', 'ugandan', 'glorified', 'lesbianism', 'expunge', 'unnatural', 'kefa', 'ssempgani', 'ensler', 'slang', 'anatomical', 'ngos', 'mukasa', 'apparatus', 'flulike', 'rescheduled', 'stub', 'accoustic']
